# LeetCode 1778 — Shortest Path in a Hidden Grid bằng Python

Link đề: https://leetcode.com/problems/shortest-path-in-a-hidden-grid/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph, Interactive

> **Ghi chú (bài Premium 🔒, interactive):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc. Đối tượng `GridMaster` thật bị LeetCode giấu; notebook tự dựng một `GridMaster` **giả lập** từ ma trận `grid` theo đúng mô tả "custom testing" của đề, có đếm số lần gọi API.

Skeleton LeetCode:

```python
# """
# This is GridMaster's API interface.
# You should not implement it, or speculate about its implementation
# """
# class GridMaster(object):
#    def canMove(self, direction: str) -> bool:
#
#    def move(self, direction: str) -> bool:
#
#    def isTarget(self) -> None:
#

class Solution(object):
    def findShortestPath(self, master: 'GridMaster') -> int:
        ...
```

## 1. Mục tiêu học

1. Làm việc với bài **interactive**: chỉ biết thế giới qua `canMove`, `move`, `isTarget`; robot **thật sự di chuyển** nên khám phá phải có **quay lui** (đi rồi đi ngược lại).
2. Tách bài thành hai pha: **(1) khám phá bản đồ bằng DFS** (vì robot chỉ đi được tới ô kề) và **(2) BFS trên bản đồ đã biết** để tìm đường ngắn nhất.
3. Hiểu vì sao không thể BFS trực tiếp bằng robot (BFS nhảy giữa các ô xa nhau).
4. Viết DFS khám phá **không đệ quy** (stack tường minh) để chịu được lưới `500 × 500`.
5. Đo **số lần gọi API** và giảm chúng bằng cách kiểm tra "đã thăm" trước khi hỏi `canMove`.

## 2. Đề bài & ví dụ

Một robot đứng trong một lưới **ẩn** kích thước `m × n`; mỗi ô **trống** hoặc **bị chặn**. Ô xuất phát và ô đích khác nhau, đều không bị chặn. Ta **không biết** kích thước lưới, vị trí xuất phát, vị trí đích. Chỉ được gọi đối tượng `GridMaster`:
- `canMove(d)`: `True` nếu robot đi được theo hướng `d`;
- `move(d)`: di chuyển robot theo hướng `d` (nếu ô đó bị chặn hoặc ra ngoài lưới thì bỏ qua, robot đứng yên);
- `isTarget()`: `True` nếu robot đang ở ô đích.

`d ∈ {'U','D','L','R'}` (lên, xuống, trái, phải). Trả về **khoảng cách ngắn nhất** (số bước) từ ô xuất phát tới ô đích, hoặc `−1` nếu không có đường.

**Custom testing:** lưới được mô tả bằng ma trận `grid`: `−1` = ô robot xuất phát, `0` = bị chặn, `1` = trống, `2` = đích (đúng một `−1` và một `2`). Code của ta **không** được đọc ma trận này.

**Ví dụ 1:** `grid = [[1,2],[-1,0]]` → `2` (đi `U` rồi `R`).

**Ví dụ 2:** `grid = [[0,0,-1],[1,1,1],[2,0,0]]` → `4`.

**Ví dụ 3:** `grid = [[-1,0],[0,2]]` → `−1`.

**Ràng buộc:** `1 ≤ m, n ≤ 500`.

In [1]:
import random
import time
import sys
from collections import deque

examples = [([[1, 2], [-1, 0]], 2), ([[0, 0, -1], [1, 1, 1], [2, 0, 0]], 4), ([[-1, 0], [0, 2]], -1)]
DIRS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}
OPP = {"U": "D", "D": "U", "L": "R", "R": "L"}

class GridMaster(object):
    """GridMaster GIẢ LẬP từ ma trận custom-testing (không phải lớp thật của LeetCode). Có đếm số lần gọi."""
    def __init__(self, grid):
        self.g = grid; self.m, self.n = len(grid), len(grid[0])
        self.r, self.c = next((i, j) for i in range(self.m) for j in range(self.n) if grid[i][j] == -1)
        self.calls = {"canMove": 0, "move": 0, "isTarget": 0}
    def _ok(self, d):
        x, y = self.r + DIRS[d][0], self.c + DIRS[d][1]
        return 0 <= x < self.m and 0 <= y < self.n and self.g[x][y] != 0
    def canMove(self, direction):
        self.calls["canMove"] += 1
        return self._ok(direction)
    def move(self, direction):
        self.calls["move"] += 1
        if self._ok(direction):
            self.r += DIRS[direction][0]; self.c += DIRS[direction][1]
            return True
        return False
    def isTarget(self):
        self.calls["isTarget"] += 1
        return self.g[self.r][self.c] == 2

def true_distance(grid):
    """Đáp án chuẩn: BFS trên ma trận thật (chỉ dùng để KIỂM TRA, lời giải không được làm vậy)."""
    m, n = len(grid), len(grid[0])
    s = next((i, j) for i in range(m) for j in range(n) if grid[i][j] == -1)
    dist = {s: 0}; q = deque([s])
    while q:
        i, j = q.popleft()
        if grid[i][j] == 2:
            return dist[(i, j)]
        for dx, dy in DIRS.values():
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and grid[x][y] != 0 and (x, y) not in dist:
                dist[(x, y)] = dist[(i, j)] + 1; q.append((x, y))
    return -1

for g, exp in examples:
    print(g, "-> khoảng cách thật:", true_distance(g), "| kỳ vọng", exp)

[[1, 2], [-1, 0]] -> khoảng cách thật: 2 | kỳ vọng 2
[[0, 0, -1], [1, 1, 1], [2, 0, 0]] -> khoảng cách thật: 4 | kỳ vọng 4
[[-1, 0], [0, 2]] -> khoảng cách thật: -1 | kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Tọa độ tương đối

Không biết vị trí thật, ta tự đặt ô xuất phát là `(0, 0)` và theo dõi tọa độ **tương đối** khi robot đi: `U` → `(x−1, y)`, `D` → `(x+1, y)`, `L` → `(x, y−1)`, `R` → `(x, y+1)`. Tọa độ có thể âm — dùng `dict`/`set` thay cho mảng.

In [2]:
pos = (0, 0)
for d in "URRD":
    pos = (pos[0] + DIRS[d][0], pos[1] + DIRS[d][1])
    print(d, "->", pos)

U -> (-1, 0)
R -> (-1, 1)
R -> (-1, 2)
D -> (0, 2)


### 3.2 Quay lui vật lý: đi rồi đi ngược lại

Robot không "dịch chuyển tức thời". Sau khi khám phá nhánh `d`, phải gọi `move(OPP[d])` để quay về ô cũ trước khi thử hướng khác.

In [3]:
gm = GridMaster(examples[0][0])
print("đầu:", (gm.r, gm.c)); gm.move("U"); print("sau U:", (gm.r, gm.c)); gm.move(OPP["U"]); print("quay lại:", (gm.r, gm.c))
print("move vào tường:", gm.move("R"), (gm.r, gm.c))

đầu: (1, 0)
sau U: (0, 0)
quay lại: (1, 0)
move vào tường: False (1, 0)


### 3.3 BFS trên tập ô đã biết

Sau khi có tập `free` (các ô đi được, tọa độ tương đối) và vị trí đích `target`, BFS từ `(0, 0)` trên `free` cho khoảng cách ngắn nhất.

### 3.4 DFS bằng stack tường minh

DFS đệ quy sâu tới 250 000 tầng sẽ vượt giới hạn đệ quy Python (mặc định 1000). Dùng stack lưu **hướng đã đi** để biết đường quay về.

In [4]:
print("giới hạn đệ quy:", sys.getrecursionlimit())

giới hạn đệ quy: 1000


## 4. Ý tưởng nền tảng / trực giác

**Vì sao không BFS thẳng bằng robot?** BFS lấy ô từ hàng đợi theo thứ tự tầng: hai ô liên tiếp trong hàng đợi có thể ở rất xa nhau ⇒ robot phải đi bộ qua lại tốn kém (và phải biết đường đi giữa chúng).

**DFS hợp với robot:** DFS luôn đi sang ô **kề** rồi quay về ô **kề** — đúng kiểu di chuyển của robot. Nhưng DFS **không** cho đường ngắn nhất.

⇒ **Kết hợp:** dùng DFS để **vẽ bản đồ** toàn bộ vùng đi được (và nhớ ô đích), rồi BFS **trên bản đồ trong bộ nhớ** (không cần robot) để lấy khoảng cách ngắn nhất.

Làm tay **Ví dụ 2** (`grid = [[0,0,-1],[1,1,1],[2,0,0]]`), robot ở `(0,2)` = tương đối `(0,0)`:
- DFS: `D` → `(1,0)` tương đối (thật `(1,2)`), `L` → `(1,−1)`, `L` → `(1,−2)`, `D` → `(2,−2)` là đích.
- Bản đồ: `{(0,0), (1,0), (1,−1), (1,−2), (2,−2)}`; BFS từ `(0,0)` tới `(2,−2)`: `4` bước.

## 5. Cách 1 — Brute Force / baseline: thử mọi đường đi độ dài `1, 2, 3, …`

**Ý tưởng:** với `L = 1, 2, …`, liệt kê (bằng DFS có quay lui vật lý) **mọi đường đi đơn** (không lặp ô) độ dài `L` từ ô xuất phát; nếu một đường kết thúc ở ô đích thì `L` là đáp án (vì các độ dài nhỏ hơn đã thử và thất bại). Dừng khi không còn đường độ dài `L` nào ⇒ `−1`.

In [5]:
def shortest_bruteforce(master):
    def dfs(depth, L, path):
        if depth == L:
            return master.isTarget(), True
        found_any = False
        for d, (dx, dy) in DIRS.items():
            x, y = path[-1][0] + dx, path[-1][1] + dy
            if (x, y) in path or not master.canMove(d):
                continue
            master.move(d); path.append((x, y))
            hit, any_ = dfs(depth + 1, L, path)
            path.pop(); master.move(OPP[d])
            found_any |= any_
            if hit:
                return True, True
        return False, found_any
    L = 1
    while True:
        hit, any_path = dfs(0, L, [(0, 0)])
        if hit:
            return L
        if not any_path:
            return -1
        L += 1

for g, exp in examples:
    gm = GridMaster(g); got = shortest_bruteforce(gm)
    print(got, "kỳ vọng", exp, "| số lần gọi:", gm.calls); assert got == exp

2 kỳ vọng 2 | số lần gọi: {'canMove': 8, 'move': 6, 'isTarget': 2}
4 kỳ vọng 4 | số lần gọi: {'canMove': 30, 'move': 20, 'isTarget': 4}
-1 kỳ vọng -1 | số lần gọi: {'canMove': 4, 'move': 0, 'isTarget': 0}


**Độ phức tạp:** số đường đi đơn tăng **theo hàm mũ** với độ dài ⇒ chỉ dùng cho lưới tí hon. Mỗi đường còn tốn các lần `move` thật.

**Nút thắt:** ta đi đi lại lại cùng một vùng rất nhiều lần. Thực ra chỉ cần **đi qua mỗi ô một lần** để biết bản đồ, rồi tính toán trên bản đồ.

## 6. Ý tưởng thuật toán / bước đột phá

**Pha 1 — DFS khám phá:** từ ô hiện tại `(x, y)`, với mỗi hướng `d`: nếu ô kề **chưa thăm** và `canMove(d)` thì `move(d)`, ghi nhận ô mới (và kiểm tra `isTarget()`), khám phá tiếp, rồi `move(OPP[d])` để quay về. Mỗi cạnh của cây DFS được đi đúng **2 lần** (đi và về).

**Pha 2 — BFS:** trên tập ô đã khám phá, BFS từ `(0, 0)`; khoảng cách tới `target` là đáp án; nếu không thấy đích trong pha 1 ⇒ `−1`.

**Quy tắc tiết kiệm API:** kiểm tra `(x, y) in visited` **trước** khi gọi `canMove` (mỗi lần gọi đều tốn).

**Invariant pha 1:** khi một lời gọi khám phá tại ô `c` kết thúc, robot **đã quay về đúng ô `c`** và mọi ô đi được, liên thông với `c` qua các ô chưa thăm, đều đã được thăm.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy khám phá + BFS

In [6]:
def explore_recursive(master):
    visited = {(0, 0)}; target = [None]
    if master.isTarget():
        target[0] = (0, 0)
    def dfs(x, y):
        for d, (dx, dy) in DIRS.items():
            nx, ny = x + dx, y + dy
            if (nx, ny) not in visited and master.canMove(d):
                visited.add((nx, ny))
                master.move(d)
                if master.isTarget():
                    target[0] = (nx, ny)
                dfs(nx, ny)
                master.move(OPP[d])
    dfs(0, 0)
    return visited, target[0]

def bfs(free, target):
    if target is None:
        return -1
    dist = {(0, 0): 0}; q = deque([(0, 0)])
    while q:
        c = q.popleft()
        if c == target:
            return dist[c]
        for dx, dy in DIRS.values():
            nb = (c[0] + dx, c[1] + dy)
            if nb in free and nb not in dist:
                dist[nb] = dist[c] + 1; q.append(nb)
    return -1

for g, exp in examples:
    gm = GridMaster(g); free, tg = explore_recursive(gm)
    print(f"bản đồ={sorted(free)}, đích={tg} -> {bfs(free, tg)} (kỳ vọng {exp}) | gọi: {gm.calls}")

bản đồ=[(-1, 0), (-1, 1), (0, 0)], đích=(-1, 1) -> 2 (kỳ vọng 2) | gọi: {'canMove': 10, 'move': 4, 'isTarget': 3}
bản đồ=[(0, 0), (1, -2), (1, -1), (1, 0), (2, -2)], đích=(2, -2) -> 4 (kỳ vọng 4) | gọi: {'canMove': 16, 'move': 8, 'isTarget': 5}
bản đồ=[(0, 0)], đích=None -> -1 (kỳ vọng -1) | gọi: {'canMove': 4, 'move': 0, 'isTarget': 1}


**Code nộp LeetCode (Cách 2):** gộp hai hàm vào `findShortestPath`; nhớ `sys.setrecursionlimit(10**6)` vì lưới tới 250 000 ô (và vẫn có rủi ro tràn ngăn xếp C) — vì vậy bản nộp dùng Cách 3.

**Độ phức tạp:** mỗi ô thăm một lần, mỗi ô hỏi tối đa 4 `canMove` ⇒ `O(N)` lời gọi API và `O(N)` thời gian, `N` = số ô đi được.

### 7.2 Cách 3 — DFS **lặp** (stack hướng đi) + BFS

**Giải thích từng đoạn:**
1. Stack chứa các khung `(x, y, chỉ số hướng tiếp theo cần thử, hướng đã đi để vào ô này)`.
2. Ở khung đỉnh: thử hướng tiếp theo; nếu đi được tới ô mới thì `move`, đẩy khung mới.
3. Khi một khung đã thử hết 4 hướng: `pop` và `move(OPP[hướng vào])` để quay về khung cha.

In [7]:
ORDER = ["U", "R", "D", "L"]

def explore_iterative(master):
    visited = {(0, 0)}
    target = (0, 0) if master.isTarget() else None
    stack = [[0, 0, 0, None]]                   # x, y, k (hướng tiếp theo), hướng đã dùng để vào
    while stack:
        top = stack[-1]
        x, y, k, came = top
        if k == 4:                             # hết hướng: quay về cha
            stack.pop()
            if came is not None:
                master.move(OPP[came])
            continue
        top[2] += 1
        d = ORDER[k]; nx, ny = x + DIRS[d][0], y + DIRS[d][1]
        if (nx, ny) in visited or not master.canMove(d):
            continue
        visited.add((nx, ny)); master.move(d)
        if master.isTarget():
            target = (nx, ny)
        stack.append([nx, ny, 0, d])
    return visited, target

for g, exp in examples:
    gm = GridMaster(g); free, tg = explore_iterative(gm)
    print(f"-> {bfs(free, tg)} (kỳ vọng {exp}) | robot đã về ô xuất phát? {(gm.r, gm.c) == next((i, j) for i, row in enumerate(g) for j, v in enumerate(row) if v == -1)} | gọi: {gm.calls}")

-> 2 (kỳ vọng 2) | robot đã về ô xuất phát? True | gọi: {'canMove': 10, 'move': 4, 'isTarget': 3}
-> 4 (kỳ vọng 4) | robot đã về ô xuất phát? True | gọi: {'canMove': 16, 'move': 8, 'isTarget': 5}
-> -1 (kỳ vọng -1) | robot đã về ô xuất phát? True | gọi: {'canMove': 4, 'move': 0, 'isTarget': 1}


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** như Cách 2 nhưng không phụ thuộc giới hạn đệ quy.

### 7.3 Cách 4 — Dừng BFS sớm và BFS hai phía trên bản đồ

Pha 2 có thể dừng ngay khi lấy `target` ra khỏi hàng đợi (Cách 2/3 đã làm). Khi bản đồ rất lớn, **BFS hai phía** (từ `(0,0)` và từ `target` xen kẽ, mỗi lần mở rộng phía có hàng đợi nhỏ hơn) thường thăm ít ô hơn nhiều.

In [8]:
def bfs_bidirectional(free, target):
    if target is None:
        return -1
    if target == (0, 0):
        return 0
    da, db = {(0, 0): 0}, {target: 0}
    qa, qb = deque([(0, 0)]), deque([target])
    while qa and qb:
        if len(qa) > len(qb):
            qa, qb, da, db = qb, qa, db, da
        for _ in range(len(qa)):                 # mở rộng trọn một tầng của phía nhỏ hơn
            c = qa.popleft()
            for dx, dy in DIRS.values():
                nb = (c[0] + dx, c[1] + dy)
                if nb in free and nb not in da:
                    da[nb] = da[c] + 1
                    if nb in db:
                        return da[nb] + db[nb]
                    qa.append(nb)
    return -1

for g, exp in examples:
    free, tg = explore_iterative(GridMaster(g))
    print(bfs_bidirectional(free, tg), "kỳ vọng", exp)

2 kỳ vọng 2
4 kỳ vọng 4
-1 kỳ vọng -1


**Độ phức tạp Cách 4:** pha 1 vẫn `O(N)` (bắt buộc — ta không biết đích ở đâu trước khi khám phá); pha 2 tệ nhất `O(N)` nhưng thường nhỏ hơn.

Mở rộng **trọn một tầng** mỗi lần là quan trọng: nếu dừng ngay khi hai phía chạm nhau giữa chừng một tầng, tổng `da + db` vẫn đúng vì mọi ô ở tầng trước đã được xử lý xong.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2**: in mỗi lần gọi `move` và tọa độ tương đối của robot.

In [9]:
class TracingMaster(GridMaster):
    def __init__(self, grid):
        super().__init__(grid); self.rel = (0, 0)
    def move(self, d):
        ok = super().move(d)
        if ok:
            self.rel = (self.rel[0] + DIRS[d][0], self.rel[1] + DIRS[d][1])
        print(f"  move({d}) -> tương đối {self.rel}, thật {(self.r, self.c)}{'  <- ĐÍCH' if self.g[self.r][self.c] == 2 else ''}")
        return ok

g, exp = examples[1]
tm = TracingMaster(g)
free, tg = explore_iterative(tm)
print("bản đồ:", sorted(free), "| đích (tương đối):", tg)
print("BFS ->", bfs(free, tg), "| số lần gọi:", tm.calls)

  move(D) -> tương đối (1, 0), thật (1, 2)
  move(L) -> tương đối (1, -1), thật (1, 1)
  move(L) -> tương đối (1, -2), thật (1, 0)
  move(D) -> tương đối (2, -2), thật (2, 0)  <- ĐÍCH
  move(U) -> tương đối (1, -2), thật (1, 0)
  move(R) -> tương đối (1, -1), thật (1, 1)
  move(R) -> tương đối (1, 0), thật (1, 2)
  move(U) -> tương đối (0, 0), thật (0, 2)
bản đồ: [(0, 0), (1, -2), (1, -1), (1, 0), (2, -2)] | đích (tương đối): (2, -2)
BFS -> 4 | số lần gọi: {'canMove': 16, 'move': 8, 'isTarget': 5}


| pha | việc | kết quả |
|---|---|---|
| 1 | DFS khám phá `D, L, L, D` rồi quay về | 5 ô, đích `(2, −2)` |
| 2 | BFS `(0,0) → (1,0) → (1,−1) → (1,−2) → (2,−2)` | **4** |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — DFS lặp khám phá + BFS.

In [10]:
# """
# This is GridMaster's API interface.
# You should not implement it, or speculate about its implementation
# """
# class GridMaster(object):
#    def canMove(self, direction: str) -> bool: ...
#    def move(self, direction: str) -> bool: ...
#    def isTarget(self) -> None: ...

class Solution(object):
    def findShortestPath(self, master: 'GridMaster') -> int:
        dirs = {"U": (-1, 0), "R": (0, 1), "D": (1, 0), "L": (0, -1)}
        opp = {"U": "D", "D": "U", "L": "R", "R": "L"}
        order = "URDL"
        visited = {(0, 0)}
        target = (0, 0) if master.isTarget() else None
        stack = [[0, 0, 0, None]]
        while stack:
            top = stack[-1]
            x, y, k, came = top
            if k == 4:
                stack.pop()
                if came is not None:
                    master.move(opp[came])
                continue
            top[2] += 1
            d = order[k]
            nx, ny = x + dirs[d][0], y + dirs[d][1]
            if (nx, ny) in visited or not master.canMove(d):
                continue
            visited.add((nx, ny))
            master.move(d)
            if master.isTarget():
                target = (nx, ny)
            stack.append([nx, ny, 0, d])
        if target is None:
            return -1
        dist = {(0, 0): 0}
        q = deque([(0, 0)])
        while q:
            c = q.popleft()
            if c == target:
                return dist[c]
            for dx, dy in dirs.values():
                nb = (c[0] + dx, c[1] + dy)
                if nb in visited and nb not in dist:
                    dist[nb] = dist[c] + 1
                    q.append(nb)
        return -1

for g, exp in examples:
    got = Solution().findShortestPath(GridMaster(g)); print(got, exp); assert got == exp

2 2
4 4
-1 -1


## 10. Vì sao thuật toán đúng

**Pha 1 (khám phá đầy đủ).** Invariant: khi khung của ô `c` bị `pop`, (a) robot đứng tại ô `c` rồi được đưa về ô cha bằng `move(OPP[came])` — hợp lệ vì ô cha là ô trống kề `c`; (b) mọi ô trống kề `c` đều đã được thăm (hoặc trước đó, hoặc trong khung của `c`). Từ (b), theo quy nạp trên khoảng cách, **mọi** ô trống liên thông với ô xuất phát đều vào `visited`. Tọa độ tương đối đúng vì mỗi `move` thành công dịch đúng một ô theo hướng đã biết (và ta chỉ `move(d)` sau khi `canMove(d)` trả `True`).

**Đích được ghi nhận:** ta gọi `isTarget()` ngay khi robot vào mỗi ô mới (và tại ô xuất phát) ⇒ nếu đích liên thông với ô xuất phát thì `target` được gán.

**Pha 2:** `visited` là đúng tập đỉnh của thành phần liên thông chứa ô xuất phát, với cạnh giữa các ô kề nhau — BFS cho khoảng cách ngắn nhất trong đồ thị không trọng số. Đường ngắn nhất trong lưới thật nằm trọn trong thành phần này ⇒ đáp án đúng. Nếu không thấy đích ⇒ đích không liên thông ⇒ `−1`. ∎

## 11. Phân tích Time / Space Complexity

Gọi `N` = số ô trống liên thông với ô xuất phát (`≤ 250 000`).

| Cách | Số lần gọi API | Time | Space |
|---|---|---|---|
| 1. Thử mọi đường độ dài L | mũ | mũ | `O(L)` |
| 2. DFS đệ quy + BFS | `≤ 4N canMove + 2(N−1) move + N isTarget` | `O(N)` | `O(N)` + ngăn xếp gọi hàm `O(N)` |
| 3. DFS lặp + BFS | như 2 | `O(N)` | `O(N)` |
| 4. + BFS hai phía | như 2 | `O(N)` | `O(N)` |

In [11]:
def random_grid(m, n, p_block, rng):
    g = [[0 if rng.random() < p_block else 1 for _ in range(n)] for _ in range(m)]
    cells = rng.sample([(i, j) for i in range(m) for j in range(n)], 2)
    (a, b), (c, d) = cells
    g[a][b], g[c][d] = -1, 2
    return g

rng = random.Random(1778)
g = random_grid(300, 300, 0.3, rng)
gm = GridMaster(g); t = time.perf_counter(); r = Solution().findShortestPath(gm); dt = time.perf_counter() - t
print(f"300x300: kết quả {r}, đúng {true_distance(g)}, {dt*1000:.0f} ms, số lần gọi {gm.calls}")

300x300: kết quả 174, đúng 174, 176 ms, số lần gọi {'canMove': 134450, 'move': 124180, 'isTarget': 62091}


## 12. Bộ test

Mọi cách được chạy trên `GridMaster` giả lập và đối chiếu với `true_distance` (BFS trên ma trận thật). Kiểm thêm: robot **luôn về lại** ô xuất phát sau pha 1 (dấu hiệu quay lui đúng), và lời giải **không bao giờ** gọi `move` vào tường (giả lập ghi nhận).

In [12]:
class StrictMaster(GridMaster):
    def move(self, d):
        assert self._ok(d), "move vào ô bị chặn / ra ngoài lưới!"
        return super().move(d)

def check(g, with_brute=False):
    exp = true_distance(g)
    start = next((i, j) for i, row in enumerate(g) for j, v in enumerate(row) if v == -1)
    for name, f in [("rec", lambda M: bfs(*explore_recursive(M))), ("iter", lambda M: bfs(*explore_iterative(M))),
                    ("bidir", lambda M: bfs_bidirectional(*explore_iterative(M))), ("submit", Solution().findShortestPath)]:
        M = StrictMaster(g); got = f(M)
        assert got == exp, (name, g, got, exp)
        assert (M.r, M.c) == start, (name, "robot không về ô xuất phát")
    if with_brute:
        assert shortest_bruteforce(StrictMaster(g)) == exp
    return exp

for g, exp in examples:
    assert check(g, with_brute=True) == exp
edge = [[[-1, 2]], [[-1], [2]], [[-1, 0, 2]], [[2, 1, 1, -1]], [[-1, 1], [1, 2]]]
print("edge:", [check(g, with_brute=True) for g in edge])
rng = random.Random(17780)
for _ in range(80):
    check(random_grid(rng.randint(1, 4), rng.randint(2, 4), 0.25, rng), with_brute=True)
for _ in range(300):
    check(random_grid(rng.randint(1, 15), rng.randint(2, 15), rng.choice([0.1, 0.3, 0.45]), rng))
print("Ví dụ, edge case, 80 lưới nhỏ (có brute force), 300 lưới vừa: OK")

edge: [1, 1, -1, 3, 2]
Ví dụ, edge case, 80 lưới nhỏ (có brute force), 300 lưới vừa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đưa robot quay về sau khi khám phá một nhánh

In [13]:
def wrong_no_backtrack(master):
    visited = {(0, 0)}; target = None
    def dfs(x, y):
        nonlocal target
        for d, (dx, dy) in DIRS.items():
            nx, ny = x + dx, y + dy
            if (nx, ny) not in visited and master.canMove(d):
                visited.add((nx, ny)); master.move(d)
                if master.isTarget(): target = (nx, ny)
                dfs(nx, ny)                         # SAI: thiếu master.move(OPP[d])
    dfs(0, 0)
    return bfs(visited, target)

g = [[1, 1, 1], [1, -1, 1], [1, 1, 2]]          # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_no_backtrack(GridMaster(g)), "| đúng:", true_distance(g))

sai : 8 | đúng: 2


**Vì sao sai:** sau khi nhánh đầu kết thúc, robot đang ở chỗ khác nhưng code tưởng nó ở `(x, y)` ⇒ `canMove` trả lời cho **vị trí thật** khác ⇒ bản đồ tương đối bị sai lệch. **Sửa:** `move(OPP[d])` sau mỗi nhánh.

### Lỗi 2 — Dừng khám phá ngay khi gặp đích rồi trả độ sâu DFS

In [14]:
def wrong_dfs_depth(master):
    visited = {(0, 0)}
    def dfs(x, y, depth):
        if master.isTarget():
            return depth
        for d, (dx, dy) in DIRS.items():
            nx, ny = x + dx, y + dy
            if (nx, ny) not in visited and master.canMove(d):
                visited.add((nx, ny)); master.move(d)
                r = dfs(nx, ny, depth + 1)
                master.move(OPP[d])
                if r != -1: return r
        return -1
    return dfs(0, 0, 0)

g = [[1, -1, 1], [1, 0, 2], [1, 1, 1]]          # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_dfs_depth(GridMaster(g)), "| đúng:", true_distance(g))

sai : 6 | đúng: 2


**Vì sao sai:** đường DFS tìm thấy đầu tiên **không** nhất thiết ngắn nhất. Ở đây DFS thử hướng theo thứ tự `U, D, L, R`: `U`, `D` bị chặn nên nó rẽ `L` rồi vòng xuống dưới và sang phải — 6 bước, trong khi đường `R, D` chỉ 2 bước. **Sửa:** khám phá hết rồi BFS.

### Lỗi 3 — Gọi `move` mà không kiểm `canMove` rồi vẫn cập nhật tọa độ

In [15]:
def wrong_assume_move(master):
    visited = {(0, 0)}; target = None
    def dfs(x, y):
        nonlocal target
        for d, (dx, dy) in DIRS.items():
            nx, ny = x + dx, y + dy
            if (nx, ny) not in visited:
                visited.add((nx, ny))                 # SAI: ghi nhận ô mà chưa biết đi được không
                if master.move(d):
                    if master.isTarget(): target = (nx, ny)
                    dfs(nx, ny); master.move(OPP[d])
    dfs(0, 0)
    return bfs(visited, target)

g = [[-1, 0, 2], [1, 0, 1], [1, 1, 1]]          # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_assume_move(GridMaster(g)), "| đúng:", true_distance(g))

sai : 2 | đúng: 6


**Vì sao sai:** ô tường bị đưa vào `visited` (tức bản đồ "đi được"), BFS đi xuyên tường ⇒ đường ngắn hơn thực tế. **Sửa:** chỉ ghi nhận ô sau khi `canMove(d)` (hoặc `move(d)`) báo thành công.

### Lỗi 4 — DFS đệ quy trên lưới lớn ⇒ `RecursionError`

In [16]:
snake = [[1] * 60 for _ in range(60)]             # lưới rỗng 60x60: DFS có thể đi sâu ~3600 tầng
snake[0][0] = -1; snake[59][59] = 2
old = sys.getrecursionlimit(); sys.setrecursionlimit(1000)
try:
    bfs(*explore_recursive(GridMaster(snake)))
    print("không lỗi")
except RecursionError as e:
    print("Bắt được RecursionError:", type(e).__name__)
finally:
    sys.setrecursionlimit(old)
print("Bản lặp:", Solution().findShortestPath(GridMaster(snake)), "| đúng:", true_distance(snake))

Bắt được RecursionError: RecursionError
Bản lặp: 118 | đúng: 118


**Vì sao sai:** độ sâu DFS có thể tới số ô (250 000). **Sửa:** DFS lặp (Cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Interactive + robot di chuyển thật ⇒ **DFS có quay lui vật lý** để khám phá; **BFS trong bộ nhớ** để tối ưu.
- Không biết kích thước/vị trí ⇒ **tọa độ tương đối** + `set`/`dict`.
- Mỗi lần gọi API đều có giá ⇒ kiểm tra `visited` trước `canMove`.
- Lưới lớn ⇒ tránh đệ quy sâu.
- Giống **489 Robot Room Cleaner** (khám phá + quay lui) và **1810 Minimum Path Cost in a Hidden Grid** (pha 2 dùng Dijkstra).

## 15. Biến thể của bài

1. **1810 Minimum Path Cost in a Hidden Grid**: mỗi lần `move` trả chi phí ô ⇒ pha 2 dùng **Dijkstra**.
2. **489 Robot Room Cleaner**: chỉ cần thăm mọi ô (không có `canMove`, robot tự báo đụng tường; có hướng quay).
3. **Không cần đường ngắn nhất, chỉ cần biết có tới được đích**: dừng khám phá khi gặp đích.
4. **Giới hạn số lần gọi API**: đếm và so sánh các chiến lược (Bài tập 5).
5. **Trả về chính đường đi** (chuỗi hướng) ngắn nhất — Bài tập 4.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Lời gọi API | Time | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi đường | Tăng dần độ dài | mũ | mũ | Kiểm chứng lưới tí hon |
| 2. DFS đệ quy + BFS | Vẽ bản đồ rồi BFS | `O(N)` | `O(N)` | Lưới nhỏ, code ngắn |
| 3. DFS lặp + BFS | Như 2, không đệ quy | `O(N)` | `O(N)` | **Bản nộp** |
| 4. + BFS hai phía | Giảm ô thăm ở pha 2 | `O(N)` | `O(N)` | Bản đồ lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[-1,1,2]]`; (b) `[[-1],[0],[2]]`; (c) `[[2,1],[1,-1]]`; (d) `[[-1,1,1],[0,0,1],[2,1,1]]`.

**Bài 2 (tính tay):** Với `grid = [[1,1,1],[1,-1,0],[2,0,0]]`, liệt kê tập ô (tọa độ tương đối) mà pha 1 khám phá được và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `rel_move(pos, d)` trả tọa độ tương đối mới sau khi đi hướng `d`.

**Bài 4 (tự viết lại):** Viết `shortest_route(master)` trả **chuỗi hướng** (ví dụ `"UR"`) của một đường ngắn nhất, hoặc `None`.

**Bài 5 (mini challenge):** Viết `explore_count(master)` trả `(số lần canMove, số lần move, số lần isTarget)` của Cách 3 và chứng minh/kiểm tra `move = 2·(N − 1)` với `N` = số ô khám phá được.

In [17]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def rel_move_todo(pos, d):
    # TODO: trả (pos[0] + DIRS[d][0], pos[1] + DIRS[d][1])
    pass

In [18]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def shortest_route(master):
    # TODO: pha 1 như Cách 3; pha 2 BFS lưu parent và hướng, rồi dựng ngược chuỗi
    pass

def explore_count(master):
    # TODO: chạy explore_iterative(master), trả số lần gọi từ master.calls
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: xác định vùng đi được. Mức 2: BFS. Mức 3: (a) `2`; (b) `−1`; (c) `2`; (d) `6` (vòng qua cột phải: `R, R, D, D, L, L`).

**Bài 2.** Mức 1: robot ở thật `(1,1)`; hướng `U` → `(0,1)`, rồi `L`, `R`, … Mức 2: tập tương đối `{(0,0), (−1,0), (−1,−1), (−1,1), (0,−1), (1,−1)}`. Mức 3: đích ở `(1,−1)`; đường `L, D` ⇒ `2`.

**Bài 3.** Mức 1: tra `DIRS[d]`. Mức 2: cộng từng thành phần. Mức 3: trả tuple.

**Bài 4.** Mức 1: BFS lưu `parent[nb] = (c, d)`. Mức 2: từ `target` đi ngược tới `(0,0)` gom hướng. Mức 3: đảo chuỗi; kiểm tra bằng cách cho robot **thật** đi theo chuỗi từ đầu và xem `isTarget()`.

**Bài 5.** Mức 1: đọc `master.calls`. Mức 2: mỗi ô (trừ ô đầu) được vào 1 lần và ra 1 lần. Mức 3: `move = 2(N−1)`, `isTarget = N`, `canMove ≤ 4N`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
b1 = [Solution().findShortestPath(GridMaster(g)) for g in [[[-1, 1, 2]], [[-1], [0], [2]], [[2, 1], [1, -1]], [[-1, 1, 1], [0, 0, 1], [2, 1, 1]]]]
print("Bài 1:", b1); assert b1 == [2, -1, 2, 6]
g2 = [[1, 1, 1], [1, -1, 0], [2, 0, 0]]
free, tg = explore_iterative(GridMaster(g2))
print("Bài 2: bản đồ", sorted(free), "đích", tg, "->", bfs(free, tg))
assert free == {(0, 0), (-1, 0), (-1, -1), (-1, 1), (0, -1), (1, -1)} and bfs(free, tg) == 2
def rel_move_todo(pos, d):
    return (pos[0] + DIRS[d][0], pos[1] + DIRS[d][1])
assert rel_move_todo((0, 0), "U") == (-1, 0) and rel_move_todo((2, 3), "L") == (2, 2)
print("Bài 3: OK")

Bài 1: [2, -1, 2, 6]
Bài 2: bản đồ [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 0), (1, -1)] đích (1, -1) -> 2
Bài 3: OK


In [20]:
# Bài 4, 5
def shortest_route(master):
    free, target = explore_iterative(master)
    if target is None:
        return None
    parent = {(0, 0): None}; q = deque([(0, 0)])
    while q:
        c = q.popleft()
        if c == target: break
        for d, (dx, dy) in DIRS.items():
            nb = (c[0] + dx, c[1] + dy)
            if nb in free and nb not in parent:
                parent[nb] = (c, d); q.append(nb)
    route = []; c = target
    while parent[c] is not None:
        c, d = parent[c]; route.append(d)
    return "".join(reversed(route))

def explore_count(master):
    free, _ = explore_iterative(master)
    return master.calls["canMove"], master.calls["move"], master.calls["isTarget"], len(free)

rng = random.Random(5)
for _ in range(150):
    g = random_grid(rng.randint(1, 10), rng.randint(2, 10), 0.3, rng)
    M = GridMaster(g); route = shortest_route(M)
    if route is None:
        assert true_distance(g) == -1
    else:
        assert len(route) == true_distance(g)
        for d in route: assert M.move(d)
        assert M.isTarget()
    cm, mv, it, N = explore_count(GridMaster(g))
    assert mv == 2 * (N - 1) and it == N and cm <= 4 * N
print("Bài 4, 5: OK | ví dụ:", shortest_route(GridMaster(examples[1][0])), explore_count(GridMaster(examples[1][0])))

Bài 4, 5: OK | ví dụ: DLLD (16, 8, 5, 5)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "khám phá bằng DFS lặp + BFS":**

```python
visited = {(0,0)}; target = (0,0) if master.isTarget() else None
stack = [[0, 0, 0, None]]
while stack:
    x, y, k, came = stack[-1]
    if k == 4:
        stack.pop()
        if came: master.move(opp[came])
        continue
    stack[-1][2] += 1
    d = "URDL"[k]; nb = (x+dx[d], y+dy[d])
    if nb in visited or not master.canMove(d): continue
    visited.add(nb); master.move(d)
    if master.isTarget(): target = nb
    stack.append([*nb, 0, d])
# BFS trên visited từ (0,0) tới target
```

**Checklist:**
- [ ] Quay robot về (`move(OPP[d])`) sau mỗi nhánh?
- [ ] Chỉ ghi nhận ô sau khi `canMove` xác nhận?
- [ ] Kiểm tra `isTarget()` ở **ô xuất phát** và mỗi ô mới?
- [ ] Đường ngắn nhất lấy từ **BFS**, không phải độ sâu DFS?
- [ ] Không dùng đệ quy sâu (lưới 500×500)?
- [ ] Trả `−1` khi không thấy đích?